### **ID 크롤러**

In [ ]:
import time
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta  # 3개월씩 이동하기 위해
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import pandas as pd
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException

# -----------------------------------------------------------------
# [!!! NEW !!!] 날짜 범위 생성 함수
# -----------------------------------------------------------------
def generate_date_chunks(start_year=2022, end_date_str="2025-11-01"):
    """ 2025-11-01부터 2022-01-01까지 거꾸로 3개월 단위 날짜 리스트를 생성합니다. """
    chunks = []
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    current_start = datetime(start_year, 1, 1)
    
    while current_start <= end_date:
        # 3개월 뒤의 마지막 날 계산 (예: 1/1 -> 3/31)
        current_end = current_start + relativedelta(months=3) - relativedelta(days=1)
        
        # 최종 날짜(11/1)를 넘지 않도록
        if current_end > end_date:
            current_end = end_date
        
        chunks.append((
            current_start.strftime("%Y-%m-%d"),
            current_end.strftime("%Y-%m-%d")
        ))
        
        # 다음 시작일 설정 (예: 4/1)
        current_start = current_end + relativedelta(days=1)
        
    return list(reversed(chunks)) # 과거로 가야 하므로 리스트를 뒤집습니다.

# -----------------------------------------------------------------
# 헬퍼 함수: (수정 없음)
# -----------------------------------------------------------------
def get_current_active_page(driver):
    try:
        element = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.page_list strong.ds_number"))
        )
        return element.text
    except Exception as e:
        print(f"  [경고] 현재 페이지 번호를 찾는 데 실패: {e}")
        return "-1"

# -----------------------------------------------------------------
# 파일 이름 정의
# -----------------------------------------------------------------
KEYS_FILENAME = "epeople_keys_list.csv"
PROGRESS_FILE = "epeople_keys_progress.txt"

# -----------------------------------------------------------------
# [!!! NEW !!!] 이어하기: 진행 상황 로드
# -----------------------------------------------------------------
def load_progress(filename):
    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8') as f:
            # { "2025-08-01_2025-10-31", "2025-05-01_2025-07-31" }
            return set(f.read().splitlines())
    return set()

# -----------------------------------------------------------------
# [!!! NEW !!!] 이어하기: 진행 상황 저장
# -----------------------------------------------------------------
def save_progress(filename, date_key):
    # 'a' (append) 모드로 파일에 씁니다.
    with open(filename, 'a', encoding='utf-8') as f:
        f.write(date_key + "\n")

# -----------------------------------------------------------------
# [!!! NEW !!!] 검색 실행 함수
# -----------------------------------------------------------------
def enter_dates_and_search(driver, start_date, end_date):
    print(f"\n>>>> 날짜 범위 {start_date} ~ {end_date} 검색 시작 <<<<")
    try:
        # 1. 날짜 입력 (send_keys는 불안정하므로 JS로 직접 값을 주입)
        driver.execute_script(f"document.getElementById('rqstStDt').value = '{start_date}';")
        driver.execute_script(f"document.getElementById('rqstEndDt').value = '{end_date}';")
        print("  날짜 입력 완료...")
        time.sleep(0.05)
        
        # 2. '검색' 버튼 클릭
        search_button = driver.find_element(By.CSS_SELECTOR, "button.btn.black")
        driver.execute_script("arguments[0].click();", search_button)
        print("  '검색' 버튼 클릭. 결과 대기 중...")
        
        # 3. [중요] 검색 결과가 로드될 때까지 대기
        # '총 4,407건' 부분이 바뀔 때까지 기다리는 것이 가장 확실합니다.
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.prog_util span.total"))
        )
        
        total_count_text = driver.find_element(By.CSS_SELECTOR, "div.prog_util span.total").text
        print(f"  검색 성공. {total_count_text}")
        return True
        
    except Exception as e:
        print(f"  [치명적 오류] 날짜 검색 실패: {e}")
        return False


# -----------------------------------------------------------------
# 1. 셀레니움 드라이버 시작 및 메인 루프
# -----------------------------------------------------------------
print("--- 스크립트 1: ID 수집기 (3개월 단위) 시작 ---")

# 1. 날짜 범위 생성 및 진행 상황 로드
all_date_chunks = generate_date_chunks()
scraped_dates = load_progress(PROGRESS_FILE)
print(f"총 {len(all_date_chunks)}개의 3개월 단위 기간 중, {len(scraped_dates)}개를 이미 완료했습니다.")

# 2. 이미 수집된 ID 목록 로드 (중복 방지용)
all_proposal_keys = set()
if os.path.exists(KEYS_FILENAME):
    try:
        df_existing_keys = pd.read_csv(KEYS_FILENAME)
        # (prplRqstNo, instRcptSn) 튜플의 집합으로 만듭니다.
        for item in df_existing_keys.itertuples():
            all_proposal_keys.add((item.prplRqstNo, item.instRcptSn))
        print(f"기존 '{KEYS_FILENAME}'에서 {len(all_proposal_keys)}개의 고유 ID를 로드했습니다.")
    except Exception as e:
        print(f"기존 '{KEYS_FILENAME}' 로드 실패: {e}")

# 3. 셀레니움 드라이버 시작
service = Service(executable_path='./chromedriver.exe')
driver = webdriver.Chrome(service=service)
driver.implicitly_wait(5) 
driver.get("https://www.epeople.go.kr/nep/prpsl/opnPrpl/opnpblPrpslList.npaid")

# 4. 메인 날짜 루프
for start_date, end_date in all_date_chunks:
    date_key = f"{start_date}_{end_date}"
    
    # [!!! 이어하기 로직 !!!]
    if date_key in scraped_dates:
        print(f"\n--- [스킵] 날짜 범위 {date_key} (이미 완료됨) ---")
        continue

    # 4-1. 날짜 입력 및 검색
    if not enter_dates_and_search(driver, start_date, end_date):
        continue # 검색 실패 시 다음 날짜 범위로

    # 4-2. 이 날짜 범위의 모든 페이지를 긁기 (기존 페이지네이션 로직)
    chunk_keys = [] # 이번 날짜 범위에서 찾은 키만 임시 저장
    
    for page_group in range(50): # 넉넉하게 50번
        current_page_number = get_current_active_page(driver)
        print(f"\n--- 페이지 그룹 {page_group + 1} (페이지 {current_page_number}~ ) 스크래핑 시작 ---")
        
        for i in range(10):
            try:
                current_page_number = get_current_active_page(driver)
                if i > 0:
                    page_to_click_str = str(int(current_page_number) + 1)
                    button = driver.find_element(By.LINK_TEXT, page_to_click_str)
                    driver.execute_script("arguments[0].scrollIntoView(true);", button)
                    time.sleep(0.05)
                    driver.execute_script("arguments[0].click();", button)
                    WebDriverWait(driver, 10).until(
                        EC.text_to_be_present_in_element(
                            (By.CSS_SELECTOR, "div.page_list strong.ds_number"), page_to_click_str
                        )
                    )
                    current_page_number = page_to_click_str 
                
                print(f"  [ {current_page_number} ] 페이지 긁는 중...")
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.tbl tbody tr")))
                rows = driver.find_elements(By.CSS_SELECTOR, "table.tbl tbody tr")
                
                for row in rows:
                    link = row.find_element(By.CSS_SELECTOR, "td.left a")
                    onclick_attr = link.get_attribute('onclick')
                    if onclick_attr and 'fn_prplDetailPage' in onclick_attr:
                        keys = onclick_attr.split("'")
                        key_pair = (keys[1], keys[3])
                        if key_pair not in all_proposal_keys:
                            all_proposal_keys.add(key_pair) # 전체 목록에도 추가
                            chunk_keys.append(key_pair)     # 이번 청크 목록에도 추가
                            
                print(f"  [ {current_page_number} ] 페이지 완료. (이번 청크 {len(chunk_keys)} / 총 {len(all_proposal_keys)}개)")
            
            except NoSuchElementException:
                print(f"  페이지 그룹의 마지막 페이지입니다. 다음 10개로 넘어갑니다.")
                break 
            except Exception as e:
                print(f"  [오류] {current_page_number} 페이지 작업 중 오류: {e}")

        try:
            next_group_button = driver.find_element(By.CSS_SELECTOR, "span.nep_p_nextBundle a")
            driver.execute_script("arguments[0].scrollIntoView(true);", next_group_button)
            time.sleep(0.05)
            driver.execute_script("arguments[0].click();", next_group_button)
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.page_list strong.ds_number")))
            print(f"  '다음 10개(>>)' 클릭 성공.")
        except NoSuchElementException:
            print(f"  '다음 10개(>>)' 버튼 없음. 이 날짜 범위({date_key}) 스크래핑 완료.")
            break 
        except Exception as e:
            print(f"\n  '다음 10개(>>)' 버튼 클릭 중 오류: {e}")
            break
            
    # 4-3. [!!! 저장 !!!] 이번 날짜 범위에서 수집한 키들을 CSV에 '추가'
    if chunk_keys:
        keys_df_chunk = pd.DataFrame(chunk_keys, columns=['prplRqstNo', 'instRcptSn'])
        keys_df_chunk.to_csv(KEYS_FILENAME, mode='a', header=not os.path.exists(KEYS_FILENAME), index=False, encoding='utf-8-sig')
        print(f"  [저장] {len(chunk_keys)}개의 새 ID를 '{KEYS_FILENAME}'에 추가했습니다.")
    
    # 4-4. [!!! 저장 !!!] 진행 상황 파일에 이 날짜 범위를 '완료'로 기록
    save_progress(PROGRESS_FILE, date_key)
    print(f"  [진행] {date_key} 범위를 완료로 기록했습니다.")
    
# 5. 모든 날짜 범위가 끝나면 드라이버 종료
driver.quit()
print(f"\n🎉 모든 날짜 범위의 ID 수집 완료! (총 {len(all_proposal_keys)}개)")
print(f"이제 'script_2_scrape_details.py'를 실행하여 상세 페이지를 긁어오세요.")

### **content 크롤러**

In [11]:
import time
import os # [!!! NEW !!!] 파일이 있는지 확인하기 위해 필요합니다.
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import pandas as pd
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException

# -----------------------------------------------------------------
# 헬퍼 함수: 상세 페이지 스크래핑 (수정 없음)
# -----------------------------------------------------------------
def scrape_detail_page_selenium(driver, prplRqstNo, instRcptSn, current_index, total_count):
    """(Selenium) 드라이버를 이용해 상세 페이지를 긁어옵니다."""
    
    detail_url = f"https://www.epeople.go.kr/nep/prpsl/opnPrpl/opnpblPrpslView.npaid?prplRqstNo={prplRqstNo}&instRcptSn={instRcptSn}"
    progress_str = f"[{current_index}/{total_count}]"

    def get_text_by_label(label_text):
        """ '제목' 이름표(strong)를 찾아 그 뒤의 텍스트(div)를 반환합니다. """
        try:
            return driver.find_element(By.XPATH, f"//strong[text()='{label_text}']/following-sibling::div[1]").text.strip()
        except NoSuchElementException:
            print(f"    [정보] {progress_str} '{label_text}' 필드를 찾을 수 없습니다. (빈 값으로 처리)")
            return None 
    
    try:
        driver.get(detail_url) 
        
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//strong[text()='제목']/following-sibling::div[1]"))
        )
        
        title = get_text_by_label("제목")
        field = get_text_by_label("분야")
        department = get_text_by_label("처리기관")
        date = get_text_by_label("신청일")
        status = get_text_by_label("추진상황")

        content = ""
        content_items = driver.find_elements(By.CSS_SELECTOR, "div.b_content div.b_conItem")
        for item in content_items:
            tit = item.find_element(By.CSS_SELECTOR, "strong.b_conTit").text.strip()
            cont = item.find_element(By.CSS_SELECTOR, "div.b_cont").text.strip()
            content += f"--- {tit} ---\n{cont}\n\n"

        data = {
            'prplRqstNo': prplRqstNo,
            'instRcptSn': instRcptSn,
            'url': detail_url, 
            'title': title,
            'field': field,
            'department': department,
            'date': date,
            'status': status,
            'content': content.strip()
        }
        
        print(f"  [성공] {prplRqstNo} 스크래핑 완료: {data['title'][:20]}... {progress_str}")
        return data
        
    except Exception as e:
        print(f"  [오류] {prplRqstNo} 스크래핑 중 예외 발생: {e} {progress_str}")
        return None

# -----------------------------------------------------------------
# 2단계: 메인 실행 로직 (대규모 수정)
# -----------------------------------------------------------------
print("--- 스크립트 2: 상세 페이지 스크래퍼 (이어하기 모드) 시작 ---")

# --- 1. 파일 이름 정의 ---
keys_filename = "epeople_keys_list.csv" # <--- 스크립트 1이 만든 ID 목록
output_filename = "국민신문고_공개제안_데이터(최종).csv" # <--- 우리가 만들 최종 파일

# --- 2. ID 목록 파일 읽기 ---
try:
    keys_df = pd.read_csv(keys_filename)
    keys_df = keys_df.drop_duplicates(subset=['prplRqstNo', 'instRcptSn'])
    total_count = len(keys_df)
    print(f"'{keys_filename}'에서 {total_count}개의 고유 ID를 읽어왔습니다.")
except FileNotFoundError:
    print(f"'{keys_filename}' 파일을 찾을 수 없습니다. 스크립트 1을 먼저 실행하세요.")
    exit()

# --- 3. [!!! NEW !!!] 이미 긁은 ID 목록 확인 ---
scraped_ids = set() # 이미 긁은 ID를 저장할 집합(set)
if os.path.exists(output_filename):
    try:
        df_existing = pd.read_csv(output_filename)
        # 'prplRqstNo' 컬럼이 존재하면, 그 안의 ID들을 scraped_ids 집합에 추가
        if 'prplRqstNo' in df_existing.columns:
            scraped_ids = set(df_existing['prplRqstNo'])
            print(f"'{output_filename}' 파일을 찾았습니다. {len(scraped_ids)}개의 항목을 이미 수집했습니다.")
            print("수집을 이어합니다...")
    except pd.errors.EmptyDataError:
        print(f"'{output_filename}' 파일이 비어있습니다. 새로 시작합니다.")
    except Exception as e:
        print(f"기존 파일 로드 중 오류 발생: {e}. 새로 시작합니다.")
else:
    print(f"'{output_filename}' 파일을 찾을 수 없습니다. 새로 시작합니다.")
    # [!!! NEW !!!] 새 파일이므로 헤더를 미리 써줍니다.
    pd.DataFrame(columns=[
        'prplRqstNo', 'instRcptSn', 'url', 'title', 'field', 
        'department', 'date', 'status', 'content'
    ]).to_csv(output_filename, index=False, encoding='utf-8-sig')


# --- 4. 2단계 전용 새 드라이버 시작 ---
service = Service(executable_path='./chromedriver.exe')
driver = webdriver.Chrome(service=service)
driver.set_page_load_timeout(30) # 30초 멈춤 방지
print("상세 페이지 스크래핑을 위해 새 드라이버를 시작합니다. (페이지 로드 타임아웃 30초)")

# --- 5. [!!! MODIFIED !!!] CSV의 모든 행을 순회하며 스크래핑 및 '즉시 저장' ---
for index, row in enumerate(keys_df.itertuples(), start=1):
    prplRqstNo = row.prplRqstNo
    instRcptSn = row.instRcptSn
    
    # [!!! NEW !!!] 이어하기 로직
    if prplRqstNo in scraped_ids:
        print(f"  [스킵] {prplRqstNo} 이미 수집됨. [{index}/{total_count}]")
        continue # 다음 ID로 바로 넘어감
    
    # ----------------------------------------------------
    # (새로운 ID만 스크래핑 실행)
    result_data = scrape_detail_page_selenium(driver, prplRqstNo, instRcptSn, index, total_count) 
    # ----------------------------------------------------
    
    if result_data:
        # [!!! NEW !!!] 메모리에 쌓지 않고 1줄씩 파일에 '추가(append)'
        df_new_row = pd.DataFrame([result_data])
        # mode='a' (append), header=False (헤더는 위에서 이미 썼음)
        df_new_row.to_csv(output_filename, mode='a', header=False, index=False, encoding='utf-8-sig')
    
    time.sleep(0.01) # 서버 보호를 위한 0.05초 휴식
 
# --- 6. 모든 작업 완료 ---
driver.quit()
print(f"\n🎉 모든 작업 완료! '{output_filename}' 파일에 최종 저장되었습니다.")

--- 스크립트 2: 상세 페이지 스크래퍼 (이어하기 모드) 시작 ---
'epeople_keys_list.csv'에서 43469개의 고유 ID를 읽어왔습니다.
'국민신문고_공개제안_데이터(최종).csv' 파일을 찾았습니다. 34338개의 항목을 이미 수집했습니다.
수집을 이어합니다...
상세 페이지 스크래핑을 위해 새 드라이버를 시작합니다. (페이지 로드 타임아웃 30초)
  [스킵] 1AB-2510-0002045 이미 수집됨. [1/43469]
  [스킵] 1AB-2510-0002041 이미 수집됨. [2/43469]
  [스킵] 1AB-2510-0001939 이미 수집됨. [3/43469]
  [스킵] 1AB-2510-0002021 이미 수집됨. [4/43469]
  [스킵] 1AB-2510-0001945 이미 수집됨. [5/43469]
  [스킵] 1AB-2510-0001925 이미 수집됨. [6/43469]
  [스킵] 1AB-2510-0001848 이미 수집됨. [7/43469]
  [스킵] 1AB-2510-0001847 이미 수집됨. [8/43469]
  [스킵] 1AB-2510-0001844 이미 수집됨. [9/43469]
  [스킵] 1AB-2510-0001775 이미 수집됨. [10/43469]
  [스킵] 1AB-2510-0001660 이미 수집됨. [11/43469]
  [스킵] 1AB-2510-0001756 이미 수집됨. [12/43469]
  [스킵] 1AB-2510-0001736 이미 수집됨. [13/43469]
  [스킵] 1AB-2510-0001723 이미 수집됨. [14/43469]
  [스킵] 1AB-2510-0001709 이미 수집됨. [15/43469]
  [스킵] 1AB-2510-0001688 이미 수집됨. [16/43469]
  [스킵] 1AB-2510-0001645 이미 수집됨. [17/43469]
  [스킵] 1AB-2510-0001613 이미 수집됨. [18/43469]
  [스킵] 1AB-2510-0001594 